# 🧱 03 · 区块结构 & Merkle 树

> 区块头 80 字节、Merkle Tree、Coinbase 交易。

## 3.1 比特币区块头 (80 bytes)

| 字段 | 大小 | 描述 |
|------|------|------|
| version | 4B | 区块版本号 |
| previous_block_hash | 32B | 前一区块的哈希（链的链接） |
| merkle_root | 32B | 所有交易的 Merkle 树根 |
| timestamp | 4B | Unix 时间戳 |
| bits | 4B | 压缩难度目标 |
| nonce | 4B | 工作证明随机数 |

**总共 80 字节**。这就是矿工翻来覆去哈希的东西。

In [ ]:
import hashlib, struct, time, random
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

@dataclass
class BlockHeader:
    version: int = 1
    prev_hash: bytes = field(default_factory=lambda: b"\x00" * 32)
    merkle_root: bytes = field(default_factory=lambda: b"\x00" * 32)
    timestamp: int = 0
    bits: int = 0x1d00ffff
    nonce: int = 0

    def serialize(self) -> bytes:
        # 80 bytes, little-endian
        return struct.pack(
            "<I32s32sIII",
            self.version,
            self.prev_hash,
            self.merkle_root,
            self.timestamp,
            self.bits,
            self.nonce,
        )

    def block_hash(self) -> bytes:
        # Double SHA-256, Bitcoin convention: display as LE
        d = hashlib.sha256(hashlib.sha256(self.serialize()).digest()).digest()
        return d[::-1]

# Test
header = BlockHeader(timestamp=int(time.time()))
print(f"Header size: {len(header.serialize())} bytes (should be 80)")
print(f"Block hash:  {header.block_hash().hex()}")


## 3.2 Merkle Tree

Merkle 树是一棵**二叉树**，每个非叶节点 = hash(左子 + 右子)。

```
           ROOT (H_ABCD)
          /            \
      H_AB              H_CD
     /    \            /    \
   H_A    H_B       H_C    H_D
   /        \       /        \
  TxA       TxB    TxC       TxD
```

**两个关键用途：**
1. **Merkle Proof**: 证明某笔交易在区块中，只需 log(n) 个哈希，不用下载整个区块
2. **SPV 钱包**: 手机钱包只需要区块头和 Merkle Proof，不需要完整区块

In [ ]:
def hash256(data: bytes) -> bytes:
    # Bitcoin's double SHA-256 for Merkle nodes
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

def build_merkle_root(txs: List[bytes]) -> bytes:
    if not txs:
        return b"\x00" * 32
    hashes = [hash256(tx) for tx in txs]
    while len(hashes) > 1:
        if len(hashes) % 2 == 1:
            hashes.append(hashes[-1])  # duplicate last
        hashes = [hash256(hashes[i] + hashes[i+1]) 
                  for i in range(0, len(hashes), 2)]
    return hashes[0]

def gen_merkle_proof(txs: List[bytes], tx_idx: int) -> List[Tuple[bytes, bool]]:
    # Generate proof path. Returns [(sibling_hash, is_right), ...]
    proof = []
    level = [hash256(tx) for tx in txs]
    idx = tx_idx

    while len(level) > 1:
        if len(level) % 2 == 1:
            level.append(level[-1])
        
        if idx % 2 == 0:  # left child
            sibling = level[idx + 1]
            proof.append((sibling, True))
        else:  # right child
            sibling = level[idx - 1]
            proof.append((sibling, False))
        
        level = [hash256(level[i] + level[i+1]) 
                 for i in range(0, len(level), 2)]
        idx //= 2
    
    return proof

def verify_merkle_proof(tx: bytes, proof: List[Tuple[bytes, bool]], root: bytes) -> bool:
    current = hash256(tx)
    for sibling, is_right in proof:
        if is_right:
            current = hash256(current + sibling)
        else:
            current = hash256(sibling + current)
    return current == root

# Demo with 8 transactions
txs = [f"Transaction_{i}".encode() for i in range(8)]
root = build_merkle_root(txs)

print(f"Transactions: {len(txs)}")
print(f"Merkle Root:  {root.hex()}")

# Prove tx #3 is in the block
target = 3
proof = gen_merkle_proof(txs, target)

print(f"\nMerkle proof for tx #{target}:")
for i, (sibling, is_right) in enumerate(proof):
    direction = "RIGHT" if is_right else "LEFT"
    print(f"  Step {i+1}: combine with {direction} sibling {sibling.hex()[:16]}...")

print(f"\nVerification: {verify_merkle_proof(txs[target], proof, root)} ✅")
print(f"Proof size: {len(proof)} hashes (vs downloading all {len(txs)} transactions)")


## 3.3 Coinbase Transaction

每个区块的**第一笔交易**永远是 coinbase（创币交易）：

- **没有输入** — 凭空创造比特币
- **输出** = 区块奖励 + 所有交易手续费
- 有一个 **coinbase 字段**可以写入任意数据（最多 100 字节）

中本聪在创世区块 coinbase 中写入了：
> The Times 03/Jan/2009 Chancellor on brink of second bailout for banks

这是对传统银行体系的无声嘲讽，也是一个不可篡改的时间戳证明。

现代矿工利用 coinbase 的 **extraNonce** 字段来扩展 nonce 搜索空间。

## 3.4 链 = 哈希指针的链表

```
Block 0  ←── Block 1  ←── Block 2  ←── Block 3
(genesis)   prev_hash     prev_hash     prev_hash
            = H(Block0)   = H(Block1)   = H(Block2)
```

修改任何一个区块 → 它的哈希变化 → 下一个区块的 prev_hash 断裂 → 整条链无效。

这就叫 **不可篡改性**: 改一笔历史交易 = 重挖它之后的所有区块。

In [ ]:
# Build a mini-chain to demonstrate linking
def mine_block(header: BlockHeader, target_zeros: int, max_tries=50_000_000) -> bool:
    target = b"\x00" * target_zeros
    for nonce in range(max_tries):
        header.nonce = nonce
        h = hashlib.sha256(hashlib.sha256(header.serialize()).digest()).digest()
        if h.startswith(target):
            return True
    return False

# Create chain
chain = []
prev_hash = b"\x00" * 32

for i in range(5):
    txs = [f"tx_{i}_{j}".encode() for j in range(random.randint(2, 4))]
    merkle = build_merkle_root(txs)

    header = BlockHeader(
        prev_hash=prev_hash,
        merkle_root=merkle,
        timestamp=int(time.time()),
    )

    success = mine_block(header, target_zeros=3)
    block_hash = header.block_hash() if success else b"FAIL"

    chain.append((header, block_hash, len(txs)))
    prev_hash = block_hash

    status = "✅" if success else "❌"
    print(f"Block {i}: {status} txs={len(txs)} hash={block_hash.hex()[:16]}... prev={header.prev_hash.hex()[:16]}...")

# Verify chain integrity
valid = all(
    chain[i][1] == chain[i+1][0].prev_hash
    for i in range(len(chain) - 1)
)
print(f"\nChain integrity: {'✅ VALID' if valid else '❌ BROKEN'}")


## 小结

✅ 区块头仅 80 字节，包括 6 个字段
✅ Merkle Tree 用 O(log n) 空间证明交易包含
✅ Coinbase = 无输入的特殊交易，矿工写入 extraNonce
✅ 链由哈希指针连接，修改历史需要重算之后所有块

**下一课**: 把所有这些拼装成完整的矿工模拟器。